# Clustering / Semantic Search

[sentence-transformers](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

In [ ]:
!pip install -U sentence-transformers
!pip install sentence-transformers scikit-learn
!pip install sentence-transformers hdbscan

In [ ]:
from sentence_transformers import SentenceTransformer
sentences = ["How do I reset my password?", "I forgot my password. How can I get back in?"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
#print(embeddings)


In [ ]:
from sentence_transformers import util
print("Shape:", embeddings.shape)

Shape: (2, 384)


In [ ]:
score = util.cos_sim(embeddings[0], embeddings[1]).item()
print(f"Cosine similarity: {score:.3f}")

Cosine similarity: 0.818


In [ ]:
print(util.cos_sim(embeddings, embeddings))

tensor([[1.0000, 0.8179],
        [0.8179, 1.0000]])


In [ ]:
from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import KMeans

# 1. Short texts to organize and search
tickets = [
    "I forgot my password and cannot log in.",
    "How can I reset my account password?",
    "My login credentials are not working.",
    "Where is my package?",
    "My order has not arrived yet.",
    "Can I track my delivery?",
    "I want a refund for my purchase.",
    "How do I return a damaged item?",
    "What is your return policy?",
]

# 2. Turn each ticket into a 384-number embedding
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(tickets, normalize_embeddings=True)

print("Embedding shape:", embeddings.shape)  # (9, 384)

# ---- CLUSTERING: discover groups of similar tickets ----
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

print("\nClusters:")
for cluster_id in range(3):
    print(f"\nCluster {cluster_id}:")
    for ticket, assigned_cluster in zip(tickets, cluster_ids):
        if assigned_cluster == cluster_id:
            print(f"  - {ticket}")

# ---- SEMANTIC SEARCH: find tickets related to a new question ----
query = "I can't access my account. How do I change my password?"
query_embedding = model.encode(query, normalize_embeddings=True)

results = util.semantic_search(
    query_embedding,
    embeddings,
    top_k=3
)[0]

print(f"\nSearch query: {query}")
for result in results:
    print(f"{result['score']:.3f}  {tickets[result['corpus_id']]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (9, 384)

Clusters:

Cluster 0:
  - Where is my package?
  - My order has not arrived yet.
  - Can I track my delivery?

Cluster 1:
  - I want a refund for my purchase.
  - How do I return a damaged item?
  - What is your return policy?

Cluster 2:
  - I forgot my password and cannot log in.
  - How can I reset my account password?
  - My login credentials are not working.

Search query: I can't access my account. How do I change my password?
0.840  How can I reset my account password?
0.742  I forgot my password and cannot log in.
0.658  My login credentials are not working.


### HDBSCAN discovers groups; it cannot guarantee that its clusters match your predefined categories

HDBSCAN = Hierarchical Density-Based Spatial Clustering of Applications with Noise

In [12]:
from sentence_transformers import SentenceTransformer
import hdbscan

tickets = [
    "I forgot my password and cannot log in.",
    "How can I reset my account password?",
    "My login credentials are not working.",
    "Where is my package?",
    "My order has not arrived yet.",
    "Can I track my delivery?",
    "I want a refund for my purchase.",
    "How do I return a damaged item?",
    "What is your return policy?",
]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(tickets, normalize_embeddings=True)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean"
)

labels = clusterer.fit_predict(embeddings)

for cluster_id in sorted(set(labels)):
    heading = "Ungrouped / noise" if cluster_id == -1 else f"Cluster {cluster_id}"
    print(f"\n{heading}")

    for ticket, label in zip(tickets, labels):
        if label == cluster_id:
            print(f"  - {ticket}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Ungrouped / noise
  - Can I track my delivery?

Cluster 0
  - I forgot my password and cannot log in.
  - How can I reset my account password?
  - My login credentials are not working.

Cluster 1
  - I want a refund for my purchase.
  - How do I return a damaged item?
  - What is your return policy?

Cluster 2
  - Where is my package?
  - My order has not arrived yet.


### Use Cassifier for known categories
If you already know the categories (e.g. Account access, Delivery, Returns, and Billing), you can use the embeddings as input to a classifier:

In [13]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

# Historical tickets that a person has already categorized
training_tickets = [
    "I forgot my password.",
    "I cannot sign in to my account.",
    "Please help me reset my login.",
    "Where is my package?",
    "My delivery is late.",
    "How can I track my order?",
    "I want to return this item.",
    "How do I get a refund?",
    "The product arrived damaged; can I send it back?",
    "I need a copy of my invoice.",
    "My payment was charged twice.",
    "Can you update my billing details?",
]

categories = [
    "Account access", "Account access", "Account access",
    "Delivery", "Delivery", "Delivery",
    "Returns", "Returns", "Returns",
    "Billing", "Billing", "Billing",
]

# Create embeddings, then train a classifier on your categories
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
training_embeddings = model.encode(training_tickets)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(training_embeddings, categories)

# Categorize a new ticket
new_ticket = "I paid twice for the same order."
new_embedding = model.encode([new_ticket])

print(classifier.predict(new_embedding)[0])  # Likely: Billing

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Billing
